# ShuffleRec: train and save the selected model

This notebook is currently set to **single-model training** using the
`compact_encoding` feature preset. Reload, restart the kernel, and Run All when
ready to train it. No optimization starts just by opening the notebook.
The selected validation weights and fitted preprocessing will be saved together
in a new private `sensitive_info/model_runs/<run>/model.pt` checkpoint.

The final test is disabled. This is a provisional usable model for completing
the recommendation workflow; it does not end future model experimentation.
Comparison mode remains available by switching RUN_TRAINING=False and
RUN_COMPARISON=True. Source CSVs and old experiments are not overwritten.


In [ ]:
import sys
from pathlib import Path
import json
from dataclasses import asdict, replace
import pandas as pd
import torch

current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, current.parent) if (p / "shufflerec").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the project root or notebooks/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shufflerec.audit import audit_paths, read_catalog, read_history, signature
from shufflerec.data import prepare_data
from shufflerec.features import FeatureConfig
from shufflerec.model import ModelConfig, SpotifySASRec
from shufflerec.training import build_evaluation_plan, evaluate, fit, seed_everything, save_checkpoint, load_checkpoint
from shufflerec.experiments import run_comparison, default_variants, individual_local_variants, estimated_duration_variants

MASTER_DB = PROJECT_ROOT / "data/master_track_table.csv"
HISTORY_DB = PROJECT_ROOT / "sensitive_info/spotify_listening_history.csv"


## Experiment controls: what changes which run?

- `RUN_TRAINING`: one fresh model using the preset named by `TRAINING_VARIANT`.
- `TRAINING_VARIANT`: choose the feature configuration to train and save. Use `custom` to use `FEATURE_CONFIG` directly.
- `RUN_COMPARISON`: independent models defined in `COMPARISON_VARIANTS`.
- `RUN_FINAL_TEST`: evaluates the single selected model on test; keep False while experimenting.
- `RUN_AUDIT`: reports coverage; it does not change the model.

**`FEATURE_CONFIG` does not override comparison variants.** Add, remove or change
entries in `COMPARISON_VARIANTS` to choose comparison groups. COMPARISON_MODE selects individual local measurements (six fits), cumulative
feature groups (four fits), or estimated-duration inclusion (two fits).
Each model starts fresh and uses the same common validation cases/candidates.
A group may also retain more training examples; coverage is included in results.

Keep candidate scope, negative count, evaluation count, sequence length and seed
fixed while comparing feature groups. There is a detailed control guide below.


In [ ]:
RUN_AUDIT = False
RUN_TRAINING = True
RUN_COMPARISON = False
RUN_FINAL_TEST = False

CANDIDATE_SCOPE = "history"
SEED = 42
MAX_LENGTH = 10
NUM_NEGATIVES = 99
MAX_EVAL_EXAMPLES = 500
MODEL_COOLDOWN_SECONDS = 10  # idle pause between models; 0 disables it

# Choose the experiment. Individual-local mode runs six fits: a common
# metadata baseline plus one fit for each local measurement.
COMPARISON_MODE = "individual_local"  # individual_local / feature_groups / estimated_duration
INCLUDE_ESTIMATED_DURATIONS = False

# Select the feature preset for the one model you want to train and save.
TRAINING_VARIANT = "compact_encoding"
# Options: audio_baseline, metadata, compact_encoding, extended_acoustics,
# no_local, add_local_rms_db, add_local_onset_strength,
# add_local_spectral_centroid, add_local_spectral_contrast,
# add_local_beat_regularity, or custom.
# FEATURE_CONFIG below is used directly only when TRAINING_VARIANT="custom".
FEATURE_CONFIG = FeatureConfig(include_estimated_durations=INCLUDE_ESTIMATED_DURATIONS)
MODEL_CONFIG = ModelConfig()

if COMPARISON_MODE == "individual_local":
    COMPARISON_VARIANTS = individual_local_variants()
elif COMPARISON_MODE == "feature_groups":
    COMPARISON_VARIANTS = default_variants()
elif COMPARISON_MODE == "estimated_duration":
    COMPARISON_VARIANTS = estimated_duration_variants()
else:
    raise ValueError("Unknown COMPARISON_MODE")

# In duration-comparison mode the two variants deliberately use opposite
# policies. Otherwise this flag applies consistently to all variants.
if COMPARISON_MODE != "estimated_duration":
    COMPARISON_VARIANTS = {
        name: replace(config, include_estimated_durations=INCLUDE_ESTIMATED_DURATIONS)
        for name, config in COMPARISON_VARIANTS.items()
    }
# Individual switches also work in FEATURE_CONFIG or a custom variant:
# FEATURE_CONFIG = replace(FEATURE_CONFIG, local_spectral_centroid=False)
# COMPARISON_VARIANTS["custom"] = replace(FEATURE_CONFIG, local_rms_db=False)
# The older `local=False` switch still disables all local measurements for
# compatibility; use local=True when selecting individual local inputs.

# Used by both single-model and comparison training. Zero preference epochs
# isolates feature effects before testing skip-preference fine-tuning.
TRAINING_OPTIONS = dict(
    epochs=5,
    preference_epochs=0,
    batch_size=32,
    learning_rate=0.001,
    preference_learning_rate=0.0001,
    patience=3,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)
if RUN_COMPARISON:
    print("Comparison configurations (no training has started):")
    print("Mode:", COMPARISON_MODE, "| fits:", len(COMPARISON_VARIANTS))
    print(pd.DataFrame([
        dict(variant=name, artist=config.artist, year=config.year, duration=config.duration,
             include_estimated_durations=config.include_estimated_durations,
             local_features=", ".join(config.enabled_local_features) or "none",
             provenance=config.provenance)
        for name, config in COMPARISON_VARIANTS.items()
    ]).to_string(index=False))


## Read-only preparation

Complete sessions are split chronologically, approximately 80/10/10 by session
count. Contexts never cross partitions. Vocabulary, aliases and numerical ranges
fit training-session tracks only. The final test is not scored here.

`history` scope loads catalog tracks appearing in the supplied history; `catalog`
uses the full supplied catalog and needs more memory. Both describe a catalog
snapshot, including held-out identities, rather than historical availability.

**Duration policy:** `INCLUDE_ESTIMATED_DURATIONS=False` excludes flagged
estimated duration values, not the entire tracks. Other enabled observed inputs
can still represent them. Set it True to include those values plus a separate
estimate indicator with its own availability flag. Invalid/missing durations
remain missing, and unflagged status remains unknown rather than certified.

`COMPARISON_MODE="estimated_duration"` deliberately compares both policies with
all other feature choices held fixed. Estimates are based on the stored history;
this switch does not reconstruct them using only pre-cutoff events. Such results
are exploratory and may contain held-out-history information. They should not
be presented as strictly chronological generalization results. The report records
this limitation. Raw duration candidates are never substituted. Policies and
selected columns are saved for consistent checkpoint/inference transformations.


In [ ]:
if CANDIDATE_SCOPE not in ("history", "catalog"):
    raise ValueError("CANDIDATE_SCOPE must be history or catalog")
if RUN_AUDIT:
    print(json.dumps(audit_paths(MASTER_DB, HISTORY_DB), indent=2))
history = read_history(HISTORY_DB)
track_ids = set(history.track_id.dropna().str.strip()) if CANDIDATE_SCOPE == "history" else None
catalog = read_catalog(MASTER_DB, track_ids=track_ids)
input_signatures = dict(master=signature(MASTER_DB), history=signature(HISTORY_DB))
print("Candidate scope:", CANDIDATE_SCOPE, "| loaded catalog tracks:", len(catalog))
data = None
model = None
if RUN_TRAINING or not RUN_COMPARISON:
    training_presets = {**default_variants(), **individual_local_variants()}
    if TRAINING_VARIANT != "custom":
        if TRAINING_VARIANT not in training_presets:
            raise ValueError("Unknown TRAINING_VARIANT. Available: " + ", ".join(sorted(training_presets)) + ", custom")
        FEATURE_CONFIG = replace(training_presets[TRAINING_VARIANT],
                                 include_estimated_durations=INCLUDE_ESTIMATED_DURATIONS)
    print("Selected training variant:", TRAINING_VARIANT)
    print("Resolved feature configuration:", json.dumps(asdict(FEATURE_CONFIG), indent=2))
    print("Active local measurements:", list(FEATURE_CONFIG.enabled_local_features))
    data = prepare_data(catalog, history, FEATURE_CONFIG, max_length=MAX_LENGTH)
    print(json.dumps(data.summary(), indent=2))
    seed_everything(SEED)
    model = SpotifySASRec(data.preprocessor, MAX_LENGTH, MODEL_CONFIG).to(DEVICE)
    print(json.dumps(model.parameter_report(), indent=2))
else:
    print("Comparison mode: each variant will prepare its own data and report counts/parameters.")


## Train and select using validation only

Both next-item and skip-pair optimization use training sessions. Validation NDCG
selects weights and supports early stopping. Preference tuning is retained only
when it improves that same validation criterion. The fixed candidate plan uses
seeded examples throughout the validation period, rather than just its first rows.
Model/preprocessing artifacts are new files; old experiments are never replaced.

**Where to choose the model:** in the settings cell, set `TRAINING_VARIANT` to
its comparison name, such as `metadata`, `compact_encoding`, or
`add_local_rms_db`. `MODEL_CONFIG` describes architecture (width, heads, etc.);
it is not the feature-preset selector. This trains fresh weights using the chosen
configuration; no comparison checkpoint path is required because comparison
fits do not save weights. The chosen name and resolved flags are printed before
training and saved in training_summary.json afterward.


In [ ]:
training_result = None
validation_plan = None
run_dir = None
if RUN_TRAINING:
    validation_plan = build_evaluation_plan(data.examples["validation"]["nip"], data.store.usable_ids,
        num_negatives=NUM_NEGATIVES, max_examples=MAX_EVAL_EXAMPLES, seed=SEED,
        scope=CANDIDATE_SCOPE + " catalog snapshot")
    training_result = fit(model, data, validation_plan, seed=SEED,
                          logger=lambda row: print(json.dumps(row)), **TRAINING_OPTIONS)
    from datetime import datetime, timezone
    from uuid import uuid4
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    run_dir = PROJECT_ROOT / "sensitive_info/model_runs" / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    metadata = dict(training=training_result, training_variant=TRAINING_VARIANT,
                    feature_config=asdict(data.preprocessor.config),
                    active_local_features=list(data.preprocessor.config.enabled_local_features),
                    partitions=data.summary(), candidate_scope=CANDIDATE_SCOPE,
                    master_signature=signature(MASTER_DB), history_signature=signature(HISTORY_DB),
                    model_parameters=model.parameter_report(), training_options=TRAINING_OPTIONS)
    save_checkpoint(run_dir / "model.pt", model, data.preprocessor, metadata=metadata)
    with (run_dir / "training_summary.json").open("x", encoding="utf-8") as stream:
        json.dump(metadata, stream, indent=2)
    print("Selected validation metrics:", training_result["best_validation"])
    print("Saved private model artifacts:", run_dir)
else:
    print("Preparation complete. Review counts and parameters before enabling RUN_TRAINING.")


## Run the selected comparison

`individual_local` (the default) runs:

| Variant | Local measurement added to the same metadata/compact-encoding baseline |
|---|---|
| `no_local` | None |
| `add_local_rms_db` | RMS level in dB |
| `add_local_onset_strength` | Onset strength |
| `add_local_spectral_centroid` | Spectral centroid |
| `add_local_spectral_contrast` | Spectral contrast |
| `add_local_beat_regularity` | Beat regularity |

Each measurement remains its own numerical input with its own availability mask.
Provenance is off in this preset to isolate measured values. If enabled in a
custom preset, only enabled local measurements contribute their own source/scope
inputs. Selecting several descriptors keeps them as separate columns, not a
combined summary. This additive comparison tests each descriptor alone; later,
a leave-one-out experiment can test their value alongside one another.

`feature_groups` preserves the four cumulative audio/metadata/encoding/acoustic
variants. `estimated_duration` compares otherwise identical configurations with
estimated durations excluded versus included with an estimate indicator.

The audio inputs still use locally filled tempo/key/mode from the merged catalog;
they are separate from the five local descriptor switches. Every variant uses
the same shared validation cases/candidates. Learning rates, batch size, epochs
and patience reach the runner. Logs and the final table identify each variant.
Reports save after the comparison finishes; comparison model weights are not saved.

After a model is released, the runner waits MODEL_COOLDOWN_SECONDS (default 10)
before preparing the next fit. It does not pause before the first or after the
last model. The delay provides idle time; it does not measure temperatures or
guarantee a particular cooling effect. Per-variant elapsed seconds exclude it.


In [ ]:
comparison_results = None
comparison_run_dir = None
if RUN_COMPARISON:
    comparison_results = run_comparison(
        catalog, history, variants=COMPARISON_VARIANTS, model_config=MODEL_CONFIG,
        max_length=MAX_LENGTH, num_negatives=NUM_NEGATIVES,
        max_examples=MAX_EVAL_EXAMPLES, seed=SEED, device=DEVICE,
        cooldown_seconds=MODEL_COOLDOWN_SECONDS,
        logger=lambda row: print(json.dumps(row)), **TRAINING_OPTIONS,
    )
    comparison_table = pd.DataFrame([
        dict(variant=r["variant"], local_features=", ".join(r["active_local_features"]) or "none",
             estimated_durations=r["uses_history_derived_duration_estimates"], hit_at_10=r["validation"]["hit_at_k"],
             ndcg_at_10=r["validation"]["ndcg_at_k"], parameters=r["parameters"]["total"],
             training_examples=r["partitions"]["train"]["nip"],
             evaluation_examples=r["validation"]["examples"],
             mean_candidates=r["validation"]["mean_candidates"],
             seconds=round(r["elapsed_seconds"], 1))
        for r in comparison_results
    ])
    print(comparison_table.to_string(index=False))

    from datetime import datetime, timezone
    from uuid import uuid4
    comparison_id = "comparison-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    comparison_run_dir = PROJECT_ROOT / "sensitive_info/model_runs" / comparison_id
    comparison_run_dir.mkdir(parents=True, exist_ok=False)
    comparison_report = dict(
        results=comparison_results, candidate_scope=CANDIDATE_SCOPE, seed=SEED,
        max_length=MAX_LENGTH, comparison_mode=COMPARISON_MODE, requested_negatives=NUM_NEGATIVES,
        max_evaluation_examples=MAX_EVAL_EXAMPLES, training_options=TRAINING_OPTIONS,
        model_cooldown_seconds=MODEL_COOLDOWN_SECONDS,
        model_config=asdict(MODEL_CONFIG), input_signatures=input_signatures,
        source_files_changed_during_run=(input_signatures != dict(master=signature(MASTER_DB), history=signature(HISTORY_DB))),
        test_evaluated=False,
    )
    with (comparison_run_dir / "comparison_results.json").open("x", encoding="utf-8") as stream:
        json.dump(comparison_report, stream, indent=2)
    print("Saved comparison metrics and epoch histories:", comparison_run_dir)


## Control reference: choosing the question for each experiment

### Feature inputs (`TRAINING_VARIANT`, custom `FEATURE_CONFIG`, or `COMPARISON_VARIANTS`)

| Setting | Meaning and useful comparison |
|---|---|
| `artist` | Artist identity; test audio baseline plus artist, or remove artist from the full model. |
| `year` | Represented release/reissue year, not necessarily original recording year. |
| `duration` | Enable the duration input; estimate inclusion is selected separately. |
| `include_estimated_durations` | Include flagged duration estimates plus a source-status indicator; exploratory when estimates use full history. |
| `genre` | Track genres; `genre` is a catalog alias fallback, not artist genres. |
| `artist_genre` | Separate artist genre labels; no effect if unavailable. |
| `local_rms_db` | Independent RMS dB input. |
| `local_onset_strength` | Independent onset-strength input. |
| `local_spectral_centroid` | Independent spectral-centroid input. |
| `local_spectral_contrast` | Independent spectral-contrast input. |
| `local_beat_regularity` | Independent beat-regularity input. |
| `local` | Legacy master gate; leave True to use individual switches. False overrides all five. |
| `provenance` | Recognized local-source flags and known analyzed seconds; can be tested independently. With local=False, applies to tempo/key/mode only. |
| `album` | Stable album ID embedding; optional, off by default. |
| `popularity` | Scaled catalog popularity; optional, off by default. Current values are not historical popularity. |
| `key_encoding` | `raw` integer code or two `circular` coordinates. |
| `time_signature_encoding` | `numeric` or a compact `categorical` embedding. |
| `max_labels` | Maximum labels retained per track/category (default 16); not sequence length. |

Original audio properties are always included. Local measurements now have individual
switches. Individual original-audio column ablations,
source-only cohorts, date windows or completion-based filters require an explicit
pipeline extension; changing MODEL_CONFIG does not select datapoints. Missing
values are handled by masks; do not zero out the master table to disable features.

Use **add-one** comparisons (audio + artist versus audio) to ask whether one
feature helps alone. Use **leave-one-out** comparisons (full versus full without
artist) to ask whether it helps alongside the others. Cumulative default variants
answer group questions; an improvement in metadata does not identify which member
of that group caused it. Keep changes small rather than running every combination.

### Data and evaluation settings

| Setting | Effect |
|---|---|
| `MASTER_DB`, `HISTORY_DB` | Choose explicit catalog/history snapshots. Source files are never rewritten. |
| `CANDIDATE_SCOPE` | `history` or full `catalog`; changes training-negative and evaluation universes, memory and difficulty. |
| `MAX_LENGTH` | Maximum preceding accepted tracks (default 10); affects context, cost and which context/target cases are comparable. |
| `NUM_NEGATIVES` | Sampled evaluation alternatives, not the number of training negatives. Larger pools usually make evaluation harder. |
| `MAX_EVAL_EXAMPLES` | Maximum validation cases sampled; does not reduce training data. |
| `SEED` | Initialization, shuffle and sampling seed. Repeat promising configurations with several seeds later. |

The 20-minute session gap and 80/10/10 split are not notebook controls today.
Changing split fractions, neutral-event rules, or history supervision changes the
experiment and should be deliberate. Shared cases are guaranteed within one
comparison call, not between arbitrary runs with different settings or variants.

### Training and model settings

| Setting | Effect |
|---|---|
| `epochs` | Maximum next-track passes; early stopping may finish sooner. |
| `preference_epochs` | Skip-pair passes; keep 0 during the initial feature comparison. |
| `learning_rate` | Next-track optimizer step size; too large can destabilize learning. |
| `preference_learning_rate` | Separate skip-phase step size; unused when preference_epochs=0. |
| `batch_size` | Examples per update; affects memory and optimization, not available features. |
| `patience` | Consecutive epochs without better validation NDCG before stopping a phase. |
| `d_model`, `nhead` | Transformer width/heads; width must divide evenly by heads. Keep 128/8 initially. |
| `feedforward`, `dropout` | Transformer capacity and dropout; keep defaults during feature comparisons. |
| `artist_dim`, `genre_dim`, `album_dim`, `time_signature_dim` | Embedding widths; affect parameter cost, not which groups are enabled. |
| `MODEL_COOLDOWN_SECONDS` | Idle seconds between model fits; default 10, 0 disables. No temperature monitoring. |
| `DEVICE` | Execution backend; CUDA when available. Different backends may not produce identical floating-point results. |

Change feature inputs first while keeping the architecture/training budget fixed.
After choosing groups, investigate context length or training settings separately.
Use validation to select configurations; keep RUN_FINAL_TEST=False until selection
is complete. Results are sampled ranking metrics, not satisfaction probabilities.


## Final test: enable only after choosing the configuration

Leave this disabled while comparing features/settings. It evaluates the selected
model on later sessions with a separate fixed candidate plan. Do not choose
another model based on this result and continue calling the same partition an
untouched test set. Current metadata snapshots cannot reconstruct all historical
availability or historical popularity.


In [ ]:
test_metrics = None
if RUN_FINAL_TEST:
    if training_result is None:
        raise ValueError("Train/select a model first, or load a chosen checkpoint explicitly")
    test_plan = build_evaluation_plan(data.examples["test"]["nip"], data.store.usable_ids,
        num_negatives=NUM_NEGATIVES, max_examples=MAX_EVAL_EXAMPLES, seed=SEED + 1,
        scope=CANDIDATE_SCOPE + " catalog snapshot")
    test_metrics = evaluate(model, data.store, test_plan)
    print(json.dumps(test_metrics, indent=2))
    with (run_dir / "final_test.json").open("x", encoding="utf-8") as stream:
        json.dump(test_metrics, stream, indent=2)


## Reusing a checkpoint

`loaded_model, fitted_preprocessor, metadata = load_checkpoint(path, device=DEVICE)`
loads the feature configuration, frozen vocabulary/scaling, architecture and
weights together. Build a `TrackStore` from the inference catalog using that
preprocessor; do not refit it. Encode past context through `store.context`, then
use `loaded_model.score_candidates` for candidate batches. Legacy 24-input
checkpoints require the legacy implementation and are not compatible here.

The notebook intentionally contains no backlog-writing utility or Spotify calls.
